# Учебное исследование территории Выборга с BlocksNet

**Что это за ноутбук.** Пошаговый учебный пример: загружаем открытые данные OpenStreetMap,
нарезаем территорию на кварталы, считаем время передвижения между кварталами и повторяем
показатели из `examples/pipeline.ipynb` библиотеки BlocksNet.

**Что получится.** Папка `examples/data/vyborg/` с GeoJSON-слоями, таблицами, графиками и
итоговым файлом `blocks_results.geojson`. Эти же файлы потом можно использовать для отдельной
HTML-страницы с интерактивной картой и дашбордом.

**Главное правило.** Ноутбук специально остановится после подготовки кварталов. Расчёты
включены только после того, как мы согласуем список метрик. Не запускайте `Run All` сразу:
лучше идти по ячейкам сверху вниз.

**Источники данных.**
- OpenStreetMap, © участники OpenStreetMap, лицензия ODbL: https://www.openstreetmap.org/
- Граница Выборга: OSM relation 1572051.
- Пешеходный граф: OpenStreetMap / Overpass API, скорость 5 км/ч.
- BlocksNet 1.0.0a9: https://github.com/aimclub/blocksnet

## 0. Как запускать

1. Установите зависимости один раз (команды есть в конце ноутбука).
2. Откройте этот ноутбук в Jupyter Lab или Jupyter Notebook.
3. Выберите ядро `.venv`.
4. Запускайте ячейки по порядку до большого блока **STOP**.
5. Посмотрите карту кварталов, напишите мне и согласуйте метрики.
6. После согласования поменяйте `RUN_CALCULATIONS = True` в ячейке с константами
   и запустите расчётные ячейки.

Если файлы уже есть, при `USE_CACHE = True` ноутбук прочитает их с диска и не будет
скачивать заново. Для принудительного обновления поставьте `USE_CACHE = False`.

In [ ]:
import importlib.metadata as md
import json
import pickle
import sys
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (10, 8)

print("Python:", sys.version.split()[0])
for package in ["blocksnet", "geopandas", "osmnx", "pandas", "numpy", "shapely", "iduedu"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: НЕ УСТАНОВЛЕН")


In [ ]:
def find_project_root(start: Path) -> Path:
    """Ищет папку, в которой лежит pyproject.toml."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "examples" / "data" / "vyborg"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
CACHE_DIR = DATA_DIR / "cache"
RESULTS_DIR = DATA_DIR / "results"

for folder in (RAW_DIR, PROCESSED_DIR, CACHE_DIR, RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# --- Параметры исследования ---
TERRITORY_LABEL = "Выборг"
TERRITORY_OSM_ID = 1572051          # OSM relation «город Выборг»
GRAPH_TYPE = "walk"                 # пешеходный граф; intermodal не используем
WALK_SPEED_KMH = 5.0                # средняя скорость пешехода
USE_CACHE = True                    # читать готовые файлы, если они есть
RUN_CALCULATIONS = False            # СТОП: включим после согласования метрик

# --- Параметры импутации (как в pipeline) ---
DEFAULT_NUMBER_OF_FLOORS = 1
LIVING_DEMAND_M2_PER_PERSON = 30
SCHOOL_SERVICE_TYPE = "school"

# --- Настройки OSMnx ---
ox.settings.use_cache = True
ox.settings.cache_folder = str(CACHE_DIR / "osmnx")
(CACHE_DIR / "osmnx").mkdir(parents=True, exist_ok=True)

print("Корень проекта:", PROJECT_ROOT)
print("Папка данных:", DATA_DIR)
print("Граф:", GRAPH_TYPE, f"{WALK_SPEED_KMH} км/ч")
print("Расчёты включены:", RUN_CALCULATIONS)


In [ ]:
def _stringify(value):
    """Приводит значения OSM/Enum к простому виду, который можно сохранить в GeoJSON."""
    if value is None:
        return None
    if hasattr(value, "value") and not isinstance(value, (str, bytes)):
        return str(value.value)
    if isinstance(value, (list, tuple)):
        return ", ".join(str(item) for item in value)
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value


def read_geojson(path: Path) -> gpd.GeoDataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Нет файла {path}. Сначала запустите ячейку, которая его создаёт."
        )
    gdf = gpd.read_file(path)
    if gdf.crs is None:
        raise ValueError(f"В файле {path} не указана система координат.")
    return gdf


def write_geojson(gdf: gpd.GeoDataFrame, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    out = gdf.copy()
    if out.crs is None:
        raise ValueError("Нельзя сохранить GeoJSON без системы координат.")
    # GeoJSON не хранит индекс. Если у таблицы есть block_id, сохраняем его колонкой.
    if out.index.name is not None:
        out = out.reset_index()
    for column in out.columns:
        if column == out.geometry.name:
            continue
        if out[column].dtype == object:
            out[column] = out[column].map(_stringify)
    numeric_columns = out.select_dtypes(include=[np.number]).columns
    if len(numeric_columns) > 0:
        out[numeric_columns] = out[numeric_columns].replace([np.inf, -np.inf], np.nan)
    out.to_crs(4326).to_file(path, driver="GeoJSON")
    print(f"Сохранено: {path}")


def load_boundary_and_crs():
    """Читает границу и возвращает её в WGS84 и в метрической проекции."""
    boundary = read_geojson(RAW_DIR / "boundary.geojson").to_crs(4326)
    metric = boundary.estimate_utm_crs()
    return boundary, boundary.to_crs(metric), metric


def ensure_block_index(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Делает block_id индексом, чтобы таблицы соединялись без путаницы."""
    if "block_id" in gdf.columns:
        gdf = gdf.set_index("block_id")
    if gdf.index.name != "block_id":
        gdf.index.name = "block_id"
    gdf.index = gdf.index.astype(int)
    return gdf


def join_extra(base: pd.DataFrame, extra: pd.DataFrame) -> pd.DataFrame:
    """Присоединяет таблицу, не создавая одинаковых колонок."""
    extra = extra.copy()
    duplicates = [column for column in extra.columns if column in base.columns]
    extra = extra.drop(columns=duplicates)
    return base.join(extra, how="left")


print("Вспомогательные функции готовы.")


## 1. Граница территории

Скачиваем границу города Выборга по OSM ID. Это не весь Выборгский район, а городское
поселение / город. Если граница выглядит неправильно, проверьте `TERRITORY_OSM_ID` в ячейке
с константами. Если вы поменяли `TERRITORY_OSM_ID`, сначала удалите папки
`examples/data/vyborg/raw` и `examples/data/vyborg/cache`: иначе ноутбук возьмёт старую
границу, старые слои OSM и старый граф.

Граница сохраняется в `raw/boundary.geojson`. Все расчёты потом идут в метрической
проекции UTM, а GeoJSON — в обычных градусах WGS84.

In [ ]:
boundary_path = RAW_DIR / "boundary.geojson"

if USE_CACHE and boundary_path.exists():
    print("Граница уже есть, читаем с диска.")
else:
    try:
        boundaries = ox.geocode_to_gdf(f"R{TERRITORY_OSM_ID}", by_osmid=True)
    except Exception as error:
        print("Не удалось загрузить границу по OSM ID:", error)
        print("Пробуем загрузить по названию.")
        boundaries = ox.geocode_to_gdf("Vyborg, Leningrad Oblast, Russia")
    boundaries = boundaries[["geometry"]].copy()
    write_geojson(boundaries, boundary_path)

boundaries, boundaries_metric, metric_crs = load_boundary_and_crs()

print("Объектов границы:", len(boundaries))
print("Метрическая система координат:", metric_crs)
print("Площадь границы, км²:", round(boundaries_metric.area.sum() / 1_000_000, 2))

fig, ax = plt.subplots(figsize=(10, 8))
boundaries_metric.plot(ax=ax, color="#cfe8f3", edgecolor="#1f4e79", linewidth=1)
ax.set_title(f"Граница территории: {TERRITORY_LABEL}")
ax.set_axis_off()
plt.show()


## 2. Дороги, железные дороги и вода

Загружаем объекты OpenStreetMap по тегам из `pipeline.ipynb`, оставляем только линии и
полигоны и сохраняем в отдельные GeoJSON. Это исходные слои для нарезки кварталов.

Если Overpass API отвечает медленно, подождите и перезапустите ячейку. Уже сохранённые
слои при `USE_CACHE = True` повторно не скачиваются.

In [ ]:
# Теги как в examples/pipeline.ipynb
bc_tags = {
    "roads": {
        "highway": [
            "construction", "crossing", "living_street", "motorway", "motorway_link",
            "motorway_junction", "pedestrian", "primary", "primary_link", "raceway",
            "residential", "road", "secondary", "secondary_link", "services", "tertiary",
            "tertiary_link", "track", "trunk", "trunk_link", "turning_circle",
            "turning_loop", "unclassified",
        ],
        "service": ["living_street", "emergency_access"],
    },
    "railways": {"railway": "rail"},
    "water": {
        "riverbank": True,
        "reservoir": True,
        "basin": True,
        "dock": True,
        "canal": True,
        "pond": True,
        "natural": ["water", "bay"],
        "waterway": ["river", "canal", "ditch"],
        "landuse": "basin",
        "water": "lake",
    },
}


def load_osm_layer(name, tags, allowed_types):
    path = RAW_DIR / f"{name}.geojson"
    if USE_CACHE and path.exists():
        layer = read_geojson(path)
        print(f"{name}: загружено из кэша, объектов {len(layer)}")
    else:
        layer = ox.features_from_polygon(
            boundaries.to_crs(4326).union_all(),
            tags=tags,
        )
        layer = layer[layer.geom_type.isin(allowed_types)].copy()
        layer = layer[["geometry"]].reset_index(drop=True)
        write_geojson(layer, path)
        print(f"{name}: скачано из OSM, объектов {len(layer)}")
    if len(layer) == 0:
        raise ValueError(f"Слой {name} пустой. Проверьте территорию и теги OSM.")
    return layer.to_crs(metric_crs)


roads = load_osm_layer("roads", bc_tags["roads"], ["LineString", "MultiLineString"])
railways = load_osm_layer("railways", bc_tags["railways"], ["LineString", "MultiLineString"])
water = load_osm_layer(
    "water",
    bc_tags["water"],
    ["Polygon", "MultiPolygon", "LineString", "MultiLineString"],
)

fig, ax = plt.subplots(figsize=(12, 10))
water.plot(ax=ax, color="#a6cee3", linewidth=0.5)
roads.plot(ax=ax, color="#333333", linewidth=0.4)
railways.plot(ax=ax, color="#d62728", linewidth=0.9)
boundaries_metric.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title("Исходные линии и вода")
ax.set_axis_off()
plt.show()


## 3. Нарезка кварталов

`preprocess_urban_objects` приводит линии и полигоны к нужному виду, а `cut_urban_blocks`
делит территорию на кварталы. Каждому кварталу присваивается устойчивый `block_id`.

Это последняя содержательная ячейка перед остановкой. После неё смотрим карту и согласуем
метрики.

In [ ]:
from blocksnet.blocks.cutting import preprocess_urban_objects, cut_urban_blocks

boundaries, boundaries_metric, metric_crs = load_boundary_and_crs()
roads = read_geojson(RAW_DIR / "roads.geojson").to_crs(metric_crs)
railways = read_geojson(RAW_DIR / "railways.geojson").to_crs(metric_crs)
water = read_geojson(RAW_DIR / "water.geojson").to_crs(metric_crs)

lines, polygons = preprocess_urban_objects(roads, railways, water)
blocks = cut_urban_blocks(boundaries_metric, lines, polygons)

blocks = blocks.reset_index(drop=True)
blocks["block_id"] = np.arange(1, len(blocks) + 1)
blocks = blocks.set_index("block_id")
blocks["site_area"] = blocks.area.astype(float)

if len(blocks) == 0:
    raise ValueError("Кварталы не получились. Проверьте исходные слои и границу.")

write_geojson(blocks, PROCESSED_DIR / "blocks_cut.geojson")

print("Кварталов:", len(blocks))
print("Суммарная площадь кварталов, км²:", round(blocks["site_area"].sum() / 1_000_000, 2))
print("Медианная площадь квартала, га:", round(blocks["site_area"].median() / 10_000, 2))

fig, ax = plt.subplots(figsize=(12, 10))
blocks.plot(ax=ax, color="#e5e5e5", edgecolor="white", linewidth=0.25)
boundaries_metric.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title(f"Кварталы территории: {TERRITORY_LABEL}")
ax.set_axis_off()
plt.show()


## 4. Функциональное использование территории (land use)

Загружаем `landuse` из OSM и назначаем кварталам преобладающее использование по правилам
из `pipeline.ipynb`. Значение `land_use` потом пригодится для карты и выводов.

Кварталы без данных получат `no_data` — это нормально для неполных данных OSM.

In [ ]:
from blocksnet.blocks.assignment import assign_land_use
from blocksnet.enums import LandUse

boundaries, boundaries_metric, metric_crs = load_boundary_and_crs()
blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "blocks_cut.geojson")).to_crs(metric_crs)

functional_zones_path = RAW_DIR / "functional_zones.geojson"
if USE_CACHE and functional_zones_path.exists():
    functional_zones = read_geojson(functional_zones_path)
    print("Функциональные зоны загружены из кэша.")
else:
    functional_zones = ox.features_from_polygon(
        boundaries.to_crs(4326).union_all(),
        tags={"landuse": True},
    ).reset_index(drop=True)
    functional_zones = functional_zones[["geometry", "landuse"]].rename(
        columns={"landuse": "functional_zone"}
    )
    functional_zones["functional_zone"] = functional_zones["functional_zone"].map(_stringify)
    write_geojson(functional_zones, functional_zones_path)
    print("Функциональные зоны скачаны из OSM.")

functional_zones = functional_zones[
    functional_zones.geom_type.isin(["Polygon", "MultiPolygon"])
].copy()

if "functional_zone" not in functional_zones.columns and "landuse" in functional_zones.columns:
    functional_zones = functional_zones.rename(columns={"landuse": "functional_zone"})

if len(functional_zones) == 0:
    raise ValueError("В OSM не найдено ни одной функциональной зоны landuse.")

functional_zones = functional_zones.to_crs(metric_crs)

rules = {
    "commercial": LandUse.BUSINESS,
    "industrial": LandUse.INDUSTRIAL,
    "cemetery": LandUse.SPECIAL,
    "garages": LandUse.INDUSTRIAL,
    "residential": LandUse.RESIDENTIAL,
    "retail": LandUse.BUSINESS,
    "grass": LandUse.RECREATION,
    "farmland": LandUse.AGRICULTURE,
    "construction": LandUse.SPECIAL,
    "brownfield": LandUse.INDUSTRIAL,
    "forest": LandUse.RECREATION,
    "recreation_ground": LandUse.RECREATION,
    "religious": LandUse.SPECIAL,
    "flowerbed": LandUse.RECREATION,
    "military": LandUse.SPECIAL,
    "landfill": LandUse.TRANSPORT,
}

blocks = assign_land_use(blocks, functional_zones, rules)

# assign_land_use оставляет только геометрию и доли land use, поэтому возвращаем
# идентификатор квартала и площадь территории.
blocks.index.name = "block_id"
blocks["site_area"] = blocks.area.astype(float)

blocks["land_use"] = blocks["land_use"].map(_stringify)
blocks["land_use"] = blocks["land_use"].fillna("no_data")

write_geojson(blocks, PROCESSED_DIR / "blocks.geojson")

print("Кварталов с land use:", int((blocks["land_use"] != "no_data").sum()), "из", len(blocks))
display(blocks["land_use"].value_counts().to_frame("кварталов"))

fig, ax = plt.subplots(figsize=(12, 10))
blocks.plot(
    column="land_use",
    legend=True,
    ax=ax,
    cmap="tab20",
    edgecolor="white",
    linewidth=0.2,
    missing_kwds={"color": "lightgrey", "label": "нет данных"},
)
boundaries_metric.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title("Функциональное использование кварталов")
ax.set_axis_off()
plt.show()


## STOP — остановка перед расчётами

**Не запускайте следующие ячейки.** Сначала посмотрите карту кварталов и проверьте:

- граница Выборга на месте;
- кварталы выглядят как кварталы, а не как случайные обрывки;
- вода и крупные дороги не «разрезали» город странным образом;
- количество кварталов и их площадь выглядят правдоподобно.

Теперь мы согласуем метрики. По умолчанию ноутбук готов повторить `pipeline.ipynb`:

- доступность (матрица времени в минутах);
- население и параметры зданий;
- вместимость школ;
- конкурентная обеспеченность школами;
- плотностные показатели `fsi`, `gsi`, `mxi`, `l`, `osr`;
- морфотипы Spacematrix и Strelka;
- area accessibility.

На остановке можно добавить, например, детские сады или поликлиники. Для этого нужно
будет добавить загрузку и агрегацию соответствующего сервиса. Пока по умолчанию оставлена
только школа, как в `pipeline.ipynb`.

Когда метрики согласованы, поменяйте `RUN_CALCULATIONS = True` в ячейке с константами
и запустите расчётные ячейки по порядку.

In [ ]:
if RUN_CALCULATIONS:
    print("Расчёты включены. Можно идти дальше.")
else:
    print("STOP: расчёты выключены.")
    print("Посмотрите карту кварталов и согласуйте метрики.")
    print("Потом поменяйте RUN_CALCULATIONS = True в ячейке с константами.")


In [ ]:
assert RUN_CALCULATIONS, (
    "STOP: расчёты отключены. Сначала посмотрите кварталы и согласуйте метрики. "
    "Потом поменяйте RUN_CALCULATIONS = True в ячейке с константами."
)
print("Продолжаем: расчёты включены.")


## 5. Пешеходный граф

Выбранный граф: **пешеходная сеть OpenStreetMap**. Источник — OpenStreetMap / Overpass API.
Способ передвижения — пешком, средняя скорость 5 км/ч. Время хранится в минутах в атрибуте
рёбер `time_min`. Интермодальный граф не строим.

BlocksNet скачивает граф через IduEdu/OSMnx и сохраняет его в `cache/walk_graph.pickle`.
При следующем запуске граф читается с диска, если совпадают территория, тип графа и скорость.

In [ ]:
from blocksnet.relations import get_accessibility_graph

boundaries, boundaries_metric, metric_crs = load_boundary_and_crs()

graph_path = CACHE_DIR / "walk_graph.pickle"
graph_metadata_path = CACHE_DIR / "metadata.json"
graph_metadata = {
    "territory_osm_id": TERRITORY_OSM_ID,
    "graph_type": GRAPH_TYPE,
    "walk_speed_kmh": WALK_SPEED_KMH,
    "created_at": pd.Timestamp.now().isoformat(),
    "blocksnet_version": md.version("blocksnet"),
}

same_cache = False
if USE_CACHE and graph_path.exists() and graph_metadata_path.exists():
    old_metadata = json.loads(graph_metadata_path.read_text(encoding="utf-8"))
    same_cache = (
        old_metadata.get("territory_osm_id") == TERRITORY_OSM_ID
        and old_metadata.get("graph_type") == GRAPH_TYPE
        and (
            GRAPH_TYPE != "walk"
            or abs(float(old_metadata.get("walk_speed_kmh", -1)) - WALK_SPEED_KMH) < 0.01
        )
    )

if same_cache:
    with open(graph_path, "rb") as file:
        graph = pickle.load(file)
    print("Граф загружен из кэша.")
else:
    graph_kwargs = {}
    if GRAPH_TYPE == "walk":
        graph_kwargs["walk_speed"] = WALK_SPEED_KMH * 1000 / 60  # метров в минуту
    graph = get_accessibility_graph(boundaries, GRAPH_TYPE, **graph_kwargs)
    with open(graph_path, "wb") as file:
        pickle.dump(graph, file, protocol=pickle.HIGHEST_PROTOCOL)
    graph_metadata_path.write_text(
        json.dumps(graph_metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Граф скачан из OSM и сохранён.")

crs_ok = graph.graph.get("crs") is not None
nodes_ok = all(("x" in data and "y" in data) for _, data in graph.nodes(data=True))
edges_ok = all("time_min" in data for _, _, data in graph.edges(data=True))

print("Узлов:", graph.number_of_nodes())
print("Рёбер:", graph.number_of_edges())
print("CRS в графе:", graph.graph.get("crs"))
print("Есть x/y у узлов:", nodes_ok)
print("Есть time_min у рёбер:", edges_ok)

assert crs_ok and nodes_ok and edges_ok, "Граф не прошёл проверку структуры."


## 6. Матрица времени между кварталами

Матрица `n × n`: на пересечении строки «квартал A» и столбца «квартал B» — время в минутах
из A в B. Диагональ около нуля. Значения `inf` означают, что маршрут не найден.

Считаем по пешеходному графу. Матрица сохраняется в CSV, потому что это таблица, а не
пространственный слой.

In [ ]:
from blocksnet.relations import calculate_accessibility_matrix

blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "blocks.geojson")).to_crs(metric_crs)
matrix_path = RESULTS_DIR / "accessibility_matrix.csv"
matrix_metadata_path = RESULTS_DIR / "accessibility_matrix_meta.json"

matrix_metadata = {
    "territory_osm_id": TERRITORY_OSM_ID,
    "graph_type": GRAPH_TYPE,
    "walk_speed_kmh": WALK_SPEED_KMH,
    "blocks_count": len(blocks),
    "created_at": pd.Timestamp.now().isoformat(),
}

same_matrix_cache = False
if USE_CACHE and matrix_path.exists() and matrix_metadata_path.exists():
    old_matrix_metadata = json.loads(matrix_metadata_path.read_text(encoding="utf-8"))
    same_matrix_cache = (
        old_matrix_metadata.get("territory_osm_id") == TERRITORY_OSM_ID
        and old_matrix_metadata.get("graph_type") == GRAPH_TYPE
        and old_matrix_metadata.get("blocks_count") == len(blocks)
        and abs(float(old_matrix_metadata.get("walk_speed_kmh", -1)) - WALK_SPEED_KMH) < 0.01
    )

if same_matrix_cache:
    acc_mx = pd.read_csv(matrix_path, index_col=0)
    acc_mx.index = acc_mx.index.astype(int)
    acc_mx.columns = acc_mx.columns.astype(int)
    acc_mx = acc_mx.reindex(index=blocks.index, columns=blocks.index)
    print("Матрица загружена из кэша.")
else:
    acc_mx = calculate_accessibility_matrix(blocks, graph, dtype=np.float32)
    acc_mx.index = acc_mx.index.astype(int)
    acc_mx.columns = acc_mx.columns.astype(int)
    acc_mx.to_csv(matrix_path, index=True)
    matrix_metadata_path.write_text(
        json.dumps(matrix_metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Матрица рассчитана и сохранена.")

assert acc_mx.shape == (len(blocks), len(blocks)), "Размер матрицы не совпадает с числом кварталов."

values = acc_mx.values
finite_values = values[np.isfinite(values)]
median_time = float(np.median(finite_values))
share_inf = float(np.isinf(values).mean())
diagonal = np.diag(values)

print("Размер матрицы:", acc_mx.shape)
print("Медианное время, минут:", round(median_time, 2))
print("Доля пар без маршрута:", round(share_inf * 100, 2), "%")
print("Диагональ: min / median / max:", round(float(np.min(diagonal)), 2), round(float(np.median(diagonal)), 2), round(float(np.max(diagonal)), 2))

median_time_by_block = pd.DataFrame(
    {"median_time_min": acc_mx.replace([np.inf], np.nan).median(axis=1)}
)
median_time_by_block.index.name = "block_id"
median_time_by_block.to_csv(RESULTS_DIR / "median_time_min.csv")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(finite_values, bins=60, color="#4c78a8", edgecolor="white")
ax.set_title("Распределение времени между кварталами")
ax.set_xlabel("Время, минут")
ax.set_ylabel("Число пар кварталов")
plt.show()

display(acc_mx.iloc[:5, :5])


## 7. Здания

Загружаем здания из OSM, определяем жилые и этажность, заполняем пропуски по правилам
`pipeline.ipynb` и агрегируем в кварталы. Получаются оценки населения и площадей.

**Важно:** это оценка по открытым данным, а не официальная статистика. Пропущенная
этажность по умолчанию заменяется на 1 этаж, а жилая площадь — на 30 м² на человека.

In [ ]:
from blocksnet.blocks.aggregation import aggregate_objects
from blocksnet.preprocessing.imputing import impute_buildings

boundaries, boundaries_metric, metric_crs = load_boundary_and_crs()
blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "blocks.geojson")).to_crs(metric_crs)

buildings_path = RAW_DIR / "buildings_raw.geojson"
if USE_CACHE and buildings_path.exists():
    buildings = read_geojson(buildings_path)
    print("Здания загружены из кэша.")
else:
    buildings = ox.features_from_polygon(
        boundaries.to_crs(4326).union_all(),
        tags={"building": True},
    )
    buildings = buildings[buildings.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    buildings = buildings.reset_index(drop=True)
    if "building" not in buildings.columns:
        buildings["building"] = "yes"
    buildings["building"] = buildings["building"].map(_stringify)
    is_living_tags = ["residential", "house", "apartments", "detached", "terrace", "dormitory"]
    buildings["is_living"] = buildings["building"].isin(is_living_tags)
    levels = (
        buildings["building:levels"]
        if "building:levels" in buildings.columns
        else pd.Series(np.nan, index=buildings.index)
    )
    buildings["number_of_floors"] = pd.to_numeric(levels, errors="coerce")
    buildings = buildings[["geometry", "building", "is_living", "number_of_floors"]]
    write_geojson(buildings, buildings_path)
    print("Здания скачаны из OSM.")

if "is_living" not in buildings.columns:
    buildings["is_living"] = False
buildings["is_living"] = buildings["is_living"].map(
    lambda value: value.strip().lower() in {"true", "1", "yes"} if isinstance(value, str) else bool(value)
)
if "number_of_floors" not in buildings.columns:
    buildings["number_of_floors"] = np.nan
buildings["number_of_floors"] = pd.to_numeric(buildings["number_of_floors"], errors="coerce")
buildings = buildings.to_crs(metric_crs)

if len(buildings) == 0:
    raise ValueError("В OSM не найдено зданий.")

buildings = impute_buildings(
    buildings,
    default_number_of_floors=DEFAULT_NUMBER_OF_FLOORS,
    default_living_demand=LIVING_DEMAND_M2_PER_PERSON,
)
write_geojson(buildings, PROCESSED_DIR / "buildings_imputed.geojson")

buildings_blocks = aggregate_objects(blocks, buildings.copy())[0]
buildings_blocks.index.name = "block_id"
buildings_blocks["site_area"] = buildings_blocks.area.astype(float)
write_geojson(buildings_blocks, PROCESSED_DIR / "buildings_blocks.geojson")

print("Зданий:", len(buildings))
print("Оценка населения:", int(round(buildings["population"].sum())))
print("Жилых зданий:", int(buildings["is_living"].sum()))

fig, ax = plt.subplots(figsize=(12, 10))
buildings_blocks.plot(
    "population",
    cmap="YlOrRd",
    legend=True,
    ax=ax,
    edgecolor="white",
    linewidth=0.2,
    missing_kwds={"color": "lightgrey"},
)
boundaries_metric.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title("Оценка населения по кварталам")
ax.set_axis_off()
plt.show()

display(buildings_blocks[["population", "footprint_area", "build_floor_area"]].describe().round(2))


## 8. Школы

Загружаем школы из OSM (`amenity=school`), заполняем пропущенную вместимость минимальным
нормативом BlocksNet и агрегируем в кварталы.

Если в OSM у школы не указана вместимость, BlocksNet подставит минимальное значение из
справочника. Это заметное допущение; в выводах его нужно упомянуть.

In [ ]:
from blocksnet.preprocessing.imputing import impute_services

boundaries, boundaries_metric, metric_crs = load_boundary_and_crs()
blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "blocks.geojson")).to_crs(metric_crs)

schools_path = RAW_DIR / "schools_raw.geojson"
if USE_CACHE and schools_path.exists():
    schools = read_geojson(schools_path)
    print("Школы загружены из кэша.")
else:
    schools = ox.features_from_polygon(
        boundaries.to_crs(4326).union_all(),
        tags={"amenity": "school"},
    ).reset_index(drop=True)
    if "capacity" in schools.columns:
        schools["capacity"] = pd.to_numeric(schools["capacity"], errors="coerce")
    else:
        schools["capacity"] = np.nan
    schools = schools[["geometry", "capacity"]]
    write_geojson(schools, schools_path)
    print("Школы скачаны из OSM.")

if len(schools) == 0:
    raise ValueError("В OSM не найдено школ. Проверьте территорию и теги.")

if "capacity" not in schools.columns:
    schools["capacity"] = np.nan

schools = schools.to_crs(metric_crs)
schools = impute_services(schools, SCHOOL_SERVICE_TYPE)
schools_blocks = aggregate_objects(blocks, schools.copy())[0]
schools_blocks.index.name = "block_id"
write_geojson(schools_blocks, PROCESSED_DIR / "schools_blocks.geojson")

print("Школ:", len(schools))
print("Суммарная вместимость, мест:", int(round(schools["capacity"].sum())))
print("Школ с нулевой вместимостью после импутации:", int((schools["capacity"] == 0).sum()))

fig, ax = plt.subplots(figsize=(12, 10))
schools_blocks.plot(
    "capacity",
    cmap="Blues",
    legend=True,
    ax=ax,
    edgecolor="white",
    linewidth=0.2,
    missing_kwds={"color": "lightgrey"},
)
boundaries_metric.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title("Вместимость школ по кварталам")
ax.set_axis_off()
plt.show()

display(schools_blocks[["capacity"]].describe().round(1))


## 9. Конкурентная обеспеченность школами

Считаем, какая доля потребности в школах покрывается с учётом конкуренции между кварталами.
`provision_strong` — обеспеченность в пределах нормативной доступности, `provision_weak` —
с учётом учреждений за её пределами.

Нормативы берём из справочника BlocksNet: для школы это 120 мест на 1000 жителей и
15 минут доступности.

In [ ]:
from blocksnet.analysis.provision import (
    competitive_provision,
    provision_strong_total,
    provision_weak_total,
)
from blocksnet.config import service_types_config

blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "blocks.geojson")).to_crs(metric_crs)
buildings_blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "buildings_blocks.geojson")).to_crs(metric_crs)
schools_blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "schools_blocks.geojson")).to_crs(metric_crs)

acc_mx = pd.read_csv(RESULTS_DIR / "accessibility_matrix.csv", index_col=0)
acc_mx.index = acc_mx.index.astype(int)
acc_mx.columns = acc_mx.columns.astype(int)
acc_mx = acc_mx.reindex(index=blocks.index, columns=blocks.index)

school_config = service_types_config[SCHOOL_SERVICE_TYPE]
school_demand = int(school_config["demand"])
school_accessibility = int(school_config["accessibility"])

provision_input = buildings_blocks[["population"]].join(schools_blocks[["capacity"]])
prov_df, links_df = competitive_provision(
    provision_input,
    acc_mx,
    school_accessibility,
    school_demand,
)

prov_df.index = prov_df.index.astype(int)
prov_df.to_csv(RESULTS_DIR / "provision.csv", index=True)
links_df.to_csv(RESULTS_DIR / "provision_links.csv", index=True)

print("Норматив: мест на 1000 жителей:", school_demand)
print("Нормативная доступность, минут:", school_accessibility)
print("Обеспеченность strong, всего:", round(float(provision_strong_total(prov_df)), 3))
print("Обеспеченность weak, всего:", round(float(provision_weak_total(prov_df)), 3))

fig, ax = plt.subplots(figsize=(12, 10))
blocks.join(prov_df)[["geometry", "provision_strong"]].plot(
    "provision_strong",
    cmap="RdYlGn",
    vmin=0,
    vmax=1,
    legend=True,
    ax=ax,
    edgecolor="white",
    linewidth=0.2,
    missing_kwds={"color": "lightgrey", "label": "нет данных"},
)
boundaries_metric.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title("Обеспеченность школами: provision_strong")
ax.set_axis_off()
plt.show()

display(prov_df[["demand", "capacity", "provision_strong", "provision_weak"]].describe().round(2))


## 10. Плотностные показатели

Считаем показатели из `pipeline.ipynb`:

- `fsi` — интенсивность использования территории;
- `gsi` — доля застроенной площади;
- `mxi` — доля жилой площади;
- `l` — средняя этажность;
- `osr` — доля открытых пространств;
- `share_living`, `share_non_living` — доли жилой и нежилой площади.

Значения безразмерные, кроме площадей.

In [ ]:
from blocksnet.analysis.indicators import calculate_density_indicators

blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "blocks.geojson")).to_crs(metric_crs)
buildings_blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "buildings_blocks.geojson")).to_crs(metric_crs)

density_df = calculate_density_indicators(buildings_blocks)
density_df.index = density_df.index.astype(int)
density_df.to_csv(RESULTS_DIR / "density.csv", index=True)

print("Плотностные показатели посчитаны.")
display(density_df[["fsi", "gsi", "mxi", "l", "osr"]].describe().round(3))

density_map = join_extra(blocks, density_df)
fig, ax = plt.subplots(figsize=(12, 10))
density_map.plot(
    "fsi",
    cmap="magma_r",
    legend=True,
    ax=ax,
    edgecolor="white",
    linewidth=0.2,
    missing_kwds={"color": "lightgrey"},
)
boundaries_metric.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title("FSI — интенсивность использования территории")
ax.set_axis_off()
plt.show()


## 11. Морфотипы Spacematrix

Spacematrix группирует застроенные кварталы по этажности, плотности и доле жилья.
Это статистическая типология, а не официальный классификатор.

Если застроенных кварталов мало, число кластеров автоматически уменьшается.

In [ ]:
from blocksnet.analysis.morphotypes import get_spacematrix_morphotypes

density_df = pd.read_csv(RESULTS_DIR / "density.csv", index_col=0)
density_df.index = density_df.index.astype(int)
blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "blocks.geojson")).to_crs(metric_crs)

# KMeans не принимает NaN и бесконечности. Оставляем кварталы с полными
# показателями и ненулевой плотностью застройки.
morph_input = density_df.replace([np.inf, -np.inf], np.nan)
morph_input = morph_input.dropna(subset=["l", "fsi", "mxi"])
morph_input = morph_input[morph_input["fsi"] > 0].copy()

developed_count = len(morph_input)
if developed_count < 2:
    raise ValueError(
        "Слишком мало застроенных кварталов с полными показателями для Spacematrix. "
        f"Их {developed_count}."
    )

n_clusters = min(11, developed_count)
spacematrix_df, spacematrix_clusters = get_spacematrix_morphotypes(
    morph_input,
    n_clusters=n_clusters,
)
spacematrix_df = spacematrix_df.rename(columns={"morphotype": "morphotype_spacematrix"})
spacematrix_df.index = spacematrix_df.index.astype(int)
spacematrix_df.to_csv(RESULTS_DIR / "morphotypes_spacematrix.csv", index=True)
spacematrix_clusters.to_csv(RESULTS_DIR / "spacematrix_clusters.csv", index=True)

print("Кластеров Spacematrix:", n_clusters)
display(spacematrix_df["morphotype_spacematrix"].value_counts().to_frame("кварталов"))

fig, ax = plt.subplots(figsize=(12, 10))
blocks.join(spacematrix_df)[["geometry", "morphotype_spacematrix"]].plot(
    "morphotype_spacematrix",
    cmap="tab20",
    legend=True,
    ax=ax,
    edgecolor="white",
    linewidth=0.2,
    missing_kwds={"color": "lightgrey", "label": "нет данных"},
)
boundaries_metric.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title("Морфотипы Spacematrix")
ax.set_axis_off()
plt.show()


## 12. Морфотипы Strelka

Strelka даёт более «городскую» типологию застройки: малоэтажная, среднеэтажная,
микрорайонная, центральная и т.п. Считаем её по тем же плотностным показателям.

In [ ]:
from blocksnet.analysis.morphotypes import get_strelka_morphotypes

strelka_input = density_df.replace([np.inf, -np.inf], np.nan)
strelka_input = strelka_input.dropna(subset=["l", "fsi", "mxi"])
strelka_input = strelka_input[strelka_input["fsi"] > 0].copy()

strelka_df = get_strelka_morphotypes(strelka_input)
strelka_df = strelka_df.rename(columns={"morphotype": "morphotype_strelka"})
strelka_df.index = strelka_df.index.astype(int)
strelka_df.to_csv(RESULTS_DIR / "morphotypes_strelka.csv", index=True)

print("Морфотипы Strelka посчитаны.")
display(strelka_df["morphotype_strelka"].value_counts().to_frame("кварталов"))

fig, ax = plt.subplots(figsize=(12, 10))
blocks.join(strelka_df)[["geometry", "morphotype_strelka"]].plot(
    "morphotype_strelka",
    cmap="tab20",
    legend=True,
    ax=ax,
    edgecolor="white",
    linewidth=0.2,
    missing_kwds={"color": "lightgrey", "label": "нет данных"},
)
boundaries_metric.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title("Морфотипы Strelka")
ax.set_axis_off()
plt.show()


## 13. Area accessibility

Показывает среднее время из квартала до всех остальных кварталов с учётом их площади.
Чем меньше значение, тем лучше доступность. Это не то же самое, что среднее по матрице:
крупные кварталы влияют сильнее.

In [ ]:
from blocksnet.analysis.network.accessibility import area_accessibility

blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "blocks.geojson")).to_crs(metric_crs)
buildings_blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "buildings_blocks.geojson")).to_crs(metric_crs)

acc_mx = pd.read_csv(RESULTS_DIR / "accessibility_matrix.csv", index_col=0)
acc_mx.index = acc_mx.index.astype(int)
acc_mx.columns = acc_mx.columns.astype(int)
acc_mx = acc_mx.reindex(index=blocks.index, columns=blocks.index)

area_acc = area_accessibility(acc_mx, buildings_blocks)
area_acc.index = area_acc.index.astype(int)
area_acc.to_csv(RESULTS_DIR / "area_accessibility.csv", index=True)

print("Area accessibility посчитана.")
display(area_acc.describe().round(2))

fig, ax = plt.subplots(figsize=(12, 10))
blocks.join(area_acc)[["geometry", "area_accessibility"]].plot(
    "area_accessibility",
    cmap="viridis_r",
    legend=True,
    ax=ax,
    edgecolor="white",
    linewidth=0.2,
    missing_kwds={"color": "lightgrey"},
)
boundaries_metric.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title("Area accessibility, минуты")
ax.set_axis_off()
plt.show()


## 14. Итоговый GeoJSON

Собираем все показатели в один слой `results/blocks_results.geojson`. Именно его потом
прочитает будущий HTML-дашборд. Одинаковые колонки не дублируются, `inf` заменяется на
пропуски.

In [ ]:
blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "blocks.geojson")).to_crs(metric_crs)
buildings_blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "buildings_blocks.geojson")).to_crs(metric_crs)
schools_blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "schools_blocks.geojson")).to_crs(metric_crs)

prov_df = pd.read_csv(RESULTS_DIR / "provision.csv", index_col=0)
prov_df.index = prov_df.index.astype(int)
density_df = pd.read_csv(RESULTS_DIR / "density.csv", index_col=0)
density_df.index = density_df.index.astype(int)
spacematrix_df = pd.read_csv(RESULTS_DIR / "morphotypes_spacematrix.csv", index_col=0)
spacematrix_df.index = spacematrix_df.index.astype(int)
strelka_df = pd.read_csv(RESULTS_DIR / "morphotypes_strelka.csv", index_col=0)
strelka_df.index = strelka_df.index.astype(int)
area_acc = pd.read_csv(RESULTS_DIR / "area_accessibility.csv", index_col=0)
area_acc.index = area_acc.index.astype(int)
median_time_by_block = pd.read_csv(RESULTS_DIR / "median_time_min.csv", index_col=0)
median_time_by_block.index = median_time_by_block.index.astype(int)

results = blocks.copy()
results = join_extra(results, buildings_blocks.drop(columns="geometry"))
results = join_extra(results, schools_blocks.drop(columns="geometry"))
results = join_extra(results, prov_df)
results = join_extra(results, density_df)
results = join_extra(results, spacematrix_df)
results = join_extra(results, strelka_df)
results = join_extra(results, area_acc)
results = join_extra(results, median_time_by_block)

for column in ["land_use", "morphotype_spacematrix", "morphotype_strelka"]:
    if column in results.columns:
        results[column] = results[column].fillna("no_data")

results.index.name = "block_id"
write_geojson(results, RESULTS_DIR / "blocks_results.geojson")

print("Строк в итоговом слое:", len(results))
print("Колонки:")
print(", ".join(map(str, results.columns)))
display(results.drop(columns="geometry").head())


## 15. Сводка и проверки

Сохраняем короткую таблицу `summary.csv` и текстовый отчёт `validation_report.txt`.
Проверки не заменяют визуальный контроль, но помогают заметить грубые ошибки.

In [ ]:
blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "blocks.geojson")).to_crs(metric_crs)
buildings_blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "buildings_blocks.geojson")).to_crs(metric_crs)
schools_blocks = ensure_block_index(read_geojson(PROCESSED_DIR / "schools_blocks.geojson")).to_crs(metric_crs)
results = read_geojson(RESULTS_DIR / "blocks_results.geojson")

acc_mx = pd.read_csv(RESULTS_DIR / "accessibility_matrix.csv", index_col=0)
acc_mx.index = acc_mx.index.astype(int)
acc_mx.columns = acc_mx.columns.astype(int)
acc_mx = acc_mx.reindex(index=blocks.index, columns=blocks.index)
prov_df = pd.read_csv(RESULTS_DIR / "provision.csv", index_col=0)
prov_df.index = prov_df.index.astype(int)

finite_values = acc_mx.values[np.isfinite(acc_mx.values)]
summary_rows = [
    ("territory", TERRITORY_LABEL),
    ("territory_osm_id", TERRITORY_OSM_ID),
    ("data_source", "OpenStreetMap / Overpass API"),
    ("graph_source", "OpenStreetMap pedestrian network"),
    ("graph_type", GRAPH_TYPE),
    ("walk_speed_kmh", WALK_SPEED_KMH),
    ("blocksnet_version", md.version("blocksnet")),
    ("blocks_count", len(blocks)),
    ("buildings_count", len(read_geojson(RAW_DIR / "buildings_raw.geojson"))),
    ("schools_count", len(read_geojson(RAW_DIR / "schools_raw.geojson"))),
    ("population_total", round(float(buildings_blocks["population"].sum()), 1)),
    ("school_capacity_total", round(float(schools_blocks["capacity"].sum()), 1)),
    ("median_travel_time_min", round(float(np.median(finite_values)), 2)),
    ("share_unreachable_pairs", round(float(np.isinf(acc_mx.values).mean()), 4)),
    ("provision_strong_mean", round(float(np.nanmean(prov_df["provision_strong"])), 4)),
    ("provision_weak_mean", round(float(np.nanmean(prov_df["provision_weak"])), 4)),
    ("area_accessibility_median_min", round(float(results["area_accessibility"].median()), 2)),
]

summary = pd.DataFrame(summary_rows, columns=["metric", "value"])
summary.to_csv(RESULTS_DIR / "summary.csv", index=False)
print("Сводка сохранена.")
display(summary)


In [ ]:
checks = []


def check(name, condition, details=""):
    checks.append((name, bool(condition), details))
    print(("OK  " if condition else "FAIL"), name, details)


check("Кварталы есть", len(blocks) > 0, f"{len(blocks)} шт.")
check("Здания есть", len(read_geojson(RAW_DIR / "buildings_raw.geojson")) > 0)
check("Школы есть", len(read_geojson(RAW_DIR / "schools_raw.geojson")) > 0)
check(
    "Размер матрицы совпадает с числом кварталов",
    acc_mx.shape == (len(blocks), len(blocks)),
    f"{acc_mx.shape}",
)
check("В матрице есть конечные значения", int(np.isfinite(acc_mx.values).sum()) > 0)
check(
    "Координаты итогового слоя заданы",
    results.crs is not None,
    str(results.crs),
)
check(
    "Число строк итогового слоя совпадает с числом кварталов",
    len(results) == len(blocks),
    f"{len(results)} / {len(blocks)}",
)

required_columns = [
    "land_use",
    "population",
    "capacity",
    "provision_strong",
    "provision_weak",
    "fsi",
    "gsi",
    "mxi",
    "l",
    "osr",
    "morphotype_spacematrix",
    "morphotype_strelka",
    "area_accessibility",
]
missing_columns = [column for column in required_columns if column not in results.columns]
check("Все основные колонки есть", not missing_columns, f"нет: {missing_columns}")

numeric_results = results.select_dtypes(include=[np.number])
check(
    "В итоговом слое нет бесконечностей",
    int(np.isinf(numeric_results).sum().sum()) == 0,
)

provision_values = prov_df["provision_strong"].dropna()
check(
    "provision_strong в диапазоне 0…1",
    len(provision_values) > 0
    and float(provision_values.min()) >= -1e-6
    and float(provision_values.max()) <= 1 + 1e-6,
    f"{float(provision_values.min()):.3f}…{float(provision_values.max()):.3f}",
)

report_lines = ["Отчёт проверки учебного исследования Выборга", ""]
for name, status, details in checks:
    report_lines.append(f"[{'OK' if status else 'FAIL'}] {name} {details}")
report_lines.append("")
report_lines.append(f"Кварталов: {len(blocks)}")
report_lines.append(f"Зданий: {len(read_geojson(RAW_DIR / 'buildings_raw.geojson'))}")
report_lines.append(f"Школ: {len(read_geojson(RAW_DIR / 'schools_raw.geojson'))}")
report_lines.append(f"Медианное время, минут: {float(np.median(finite_values)):.2f}")
report_lines.append(f"Доля пар без маршрута: {float(np.isinf(acc_mx.values).mean()) * 100:.2f}%")

report_path = RESULTS_DIR / "validation_report.txt"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print("Отчёт сохранён:", report_path)

failed = [name for name, status, _ in checks if not status]
if failed:
    raise AssertionError("Не прошли проверки: " + ", ".join(failed))


## 16. Как читать результаты

- Сравните кварталы с низкой `provision_strong` и низкой `area_accessibility`: где школа
  далеко или мест не хватает?
- Посмотрите, какие морфотипы преобладают в центре и на окраинах.
- Сопоставьте `land_use` с плотностью и населением.
- Помните об ограничениях: OSM неполный, население и вместимость школ — оценки,
  пешеходная скорость усреднена, граф не учитывает закрытые территории.

## 17. Повторный запуск

**Первая установка (один раз).** Откройте PowerShell и выполните:

```powershell
cd "D:\ai\python-package-boilerplate-main\python-package-boilerplate-main"
& "D:\ai\python-package-boilerplate-main\.venv\Scripts\Activate.ps1"
python -m pip install --upgrade pip
python -m pip install "blocksnet[ipynb]==1.0.0a9" "osmnx>=2.0,<3.0" jupyterlab
jupyter lab
```

Зависимости также записаны в `pyproject.toml`, чтобы версии не потерялись.

Если PowerShell не разрешает активацию окружения, выполните один раз:

```powershell
Set-ExecutionPolicy -Scope CurrentUser RemoteSigned
```

В первой ячейке ноутбука должно быть напечатано `blocksnet: 1.0.0a9`. Если версия другая,
значит установка не применилась: проверьте, что Jupyter запущен из того же окружения `.venv`.

**Обычный повторный запуск без скачивания:**

1. Оставьте `USE_CACHE = True`.
2. Запустите ячейки сверху вниз.
3. Граница, OSM-слои, граф и матрица прочитаются с диска.

**Только пересчёт показателей:**

1. Поменяйте `RUN_CALCULATIONS = True`.
2. Запустите ячейки начиная с раздела 9 (или с нужного показателя).

**Если меняли территорию или тип графа:** сначала удалите `raw` и `cache`, потом
запустите ноутбук сверху вниз.

**Принудительно обновить OSM-данные:**

```powershell
Remove-Item -Recurse -Force "examples\data\vyborg\raw"
```

**Обновить граф и матрицу:**

```powershell
Remove-Item -Force "examples\data\vyborg\cache\walk_graph.pickle"
Remove-Item -Force "examples\data\vyborg\results\accessibility_matrix.csv"
```

**Полностью очистить результаты:**

```powershell
Remove-Item -Recurse -Force "examples\data\vyborg\processed"
Remove-Item -Recurse -Force "examples\data\vyborg\results"
```

После удаления запустите ноутбук сверху вниз. Папки создадутся автоматически.

## 18. Что дальше

После выполнения ноутбука отдельным этапом сделаем Python-файл, который прочитает
`results/blocks_results.geojson` и CSV-файлы и соберёт одну HTML-страницу:

- интерактивная карта кварталов;
- те же показатели;
- графики и таблицы;
- короткие выводы;
- возможность смотреть разные слои.

Сейчас этот этап не выполняется: сначала нужно получить и проверить данные ноутбука.